## Initialization

In [1]:
from active_learning.al import AL
from utils.utils import get_tasks

# exp selection
# exp_name = 'acidic_mor_PtRuSc'
# exp_name = 'MOR_5D'
# exp_name = 'DFFC_PdPtCu'
exp_name = 'DFFC_8D_4'

## Save and load

In [2]:
# load al instance from database
al = AL(exp_name)

In [ ]:
# save the current al instance to the database
al.save_exp_to_db()

## AL main workflow

### recipe generation

In [4]:
%%time
# generate next trial using Bayesian Optimization, specify beta if needed, e.g. beta=100 for exploration, beta=0.2 for default
al.generate_botorch_trial(n=20, aquisition = 'qUCB', optimizer_kwargs = {'options':{'maxiter':20000000}, 'sequential':True})

BO trial successfully added as trial 16
CPU times: total: 24min 11s
Wall time: 6min 14s


In [ ]:
# generate next trial using SOBOL method (random select)
al.generate_sobol_trial(n=20)

In [6]:
# update the recipe table on the database, if trial index is not specified, the recipe of the latest trial will be uploaded
al.update_recipe_table(trial_index=None, add_pred=True)

### sample preparation

In [ ]:
# benchmark specification
benchmark = None
# benchmark={5: '1_0'}

In [ ]:
# command opentrons to prepare the sample for the latest trial, following the ot2_run_configs of the exp
al.make_sample(trial_index=None, benchmark=benchmark, operator_name='chu')

In [ ]:
# update the sample table on the database, if trial index is not specified, the sample of the latest trial will be uploaded. Note that sample table calculation is based on local ot2_run_configs.py under project folder, not the one on ot2 server.
al.update_sample_table(trial_index=None, benchmark=benchmark)

### sample testing complete

In [ ]:
# run this cell after sample testing and data analysis is completed, then go back to the BO step
al.mark_trial_complete(trial_index=None)

## AL monitor

In [ ]:
# update the exp with the latest result
al.update_exp_result()

In [5]:
# exp monitor for windows
al.exp_monitor()

,arm_name,trial_index,arm_index,Cr,Pd,Pt,Cu,Au,Ir,Ce,Nb,pred_mean,pred_std,max_power,trial_status,generation_method
0,0_0,0,0,0.382,0.202,0.020,0.007,0.095,0.056,0.146,0.092,30.8500,3.581,30.880,COMPLETED,Sobol
1,0_1,0,1,0.035,0.110,0.052,0.172,0.115,0.122,0.121,0.273,15.0200,3.237,11.420,COMPLETED,Sobol
10,0_10,0,10,0.048,0.404,0.050,0.136,0.003,0.327,0.026,0.006,43.9600,3.902,53.110,COMPLETED,Sobol
11,0_11,0,11,0.096,0.229,0.253,0.037,0.221,0.014,0.049,0.101,34.0100,3.164,27.720,COMPLETED,Sobol
12,0_12,0,12,0.036,0.008,0.081,0.249,0.028,0.175,0.213,0.210,-0.7822,3.947,1.007,COMPLETED,Sobol
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,16_5,16,5,0.000,0.570,0.210,0.000,0.107,0.000,0.089,0.024,66.8000,1.831,NaN,RUNNING,BoTorch
312,16_6,16,6,0.000,0.572,0.223,0.000,0.026,0.000,0.094,0.086,66.6600,1.611,NaN,RUNNING,BoTorch
313,16_7,16,7,0.000,0.426,0.191,0.000,0.000,0.044,0.179,0.159,66.7200,1.821,NaN,RUNNING,BoTorch
314,16_8,16,8,0.000,0.509,0.216,0.000,0.000,0.044,0.141,0.089,66.9700,1.135,NaN,RUNNING,BoTorch


In [ ]:
# exp monitor for mac and linux
from IPython.display import display, HTML
display(HTML(al.exp_monitor().to_html()))

In [ ]:
# check abandon reason
al.exp.trials[0].abandoned_reason

## AL manual edit

In [ ]:
# update the exp with the latest result and force match previous trials with the results in the database
al.update_exp_result(force_update_trial=[0, 1])

In [4]:
# manually add a trial
trial_df = get_tasks(exp_name, task_name='chia-model')
al.add_manual_trial(trial_df)

        Pd      Pt      Cu      Au      Ir      Ce      Nb      Cr
0   0.0647  0.0000  0.0000  0.0000  0.3421  0.0000  0.3550  0.2382
1   0.0000  0.0000  0.0088  0.0429  0.0000  0.2434  0.6548  0.0501
2   0.0136  0.0624  0.0000  0.0000  0.0000  0.2210  0.2560  0.4469
3   0.2363  0.0000  0.0000  0.0744  0.5677  0.1216  0.0000  0.0000
4   0.0000  0.0000  0.0111  0.0000  0.0000  0.5818  0.0900  0.3171
5   0.2322  0.1331  0.0000  0.0000  0.3304  0.0000  0.1759  0.1284
6   0.0000  0.0000  0.0000  0.0000  0.0000  0.2575  0.0756  0.6670
7   0.0000  0.0000  0.0748  0.0000  0.0920  0.0000  0.4180  0.4152
8   0.3656  0.1914  0.0000  0.0000  0.1168  0.0000  0.3178  0.0083
9   0.0000  0.0000  0.0000  0.0000  0.0000  0.7171  0.0721  0.2108
10  0.3651  0.2373  0.0405  0.0000  0.0000  0.2328  0.0000  0.1243
11  0.0975  0.1382  0.2960  0.0446  0.0000  0.1311  0.0000  0.2927
12  0.1051  0.0761  0.0107  0.2354  0.0000  0.0146  0.5581  0.0000
13  0.0541  0.0327  0.3370  0.3315  0.1109  0.0000  0.1338  0.

In [ ]:
# abandon an certain arm in a trial
al.abandon_arm(0, '0_18', 'abandoned in manual test')

In [ ]:
# abandon a whole trial
al.abandon_trial(9, 'test trial')

## Plotting

In [ ]:
from ax.modelbridge.cross_validation import cross_validate
from ax.plot.contour import interact_contour
from ax.plot.diagnostic import interact_cross_validation
from ax.plot.scatter import interact_fitted
from ax.plot.slice import interact_slice
from ax.utils.notebook.plotting import render, init_notebook_plotting
init_notebook_plotting()

In [ ]:
model = al.get_bo_model()

In [ ]:
# Contour plots
render(interact_contour(model=model, metric_name='max_power'))

In [ ]:
# Cross-validation plots
cv_results = cross_validate(model)
print('current r_squared is: ', al.calculate_r_squared(cv_results))
render(interact_cross_validation(cv_results))

In [ ]:
# Slice plots
render(interact_slice(model))

In [ ]:
# Tile plots
render(interact_fitted(model, rel=False))

## Danger Zone

In [ ]:
# delete the current active learning process from the database
al.delete_exp_on_db()

In [ ]:
# only use this if you want to start a new experiment, note that the previous experiment has to be manually deleted, first, because the script refuses to overwrite the previous experiment by default
al = AL(exp_name, load_from_db=False)